# Austin Animal Center EDA

Lightweight exploratory notebook for the course project proposal. It uses the processed modeling dataset built from the intake/outcome pipeline.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT / 'src'))

from dataset_builder import build_dataset

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

In [ ]:
processed_path = ROOT / 'data' / 'processed' / 'modeling_dataset.csv'
if not processed_path.exists():
    build_dataset(raw_dir=ROOT / 'data' / 'raw', processed_dir=ROOT / 'data' / 'processed', download=False)

df = pd.read_csv(processed_path, parse_dates=['intake_datetime', 'outcome_datetime'])
df.head()

## Dataset Overview

In [ ]:
summary = {
    'rows': len(df),
    'unique_animal_ids': df['animal_id'].nunique(),
    'fast_adoption_rate': df['fast_adoption_30d'].mean(),
    'adoption_rate': df['adoption'].mean(),
}
pd.Series(summary)

## Label Distribution

In [ ]:
ax = sns.countplot(data=df, x='fast_adoption_30d', palette='Set2')
ax.set_title('Fast Adoption Within 30 Days')
ax.set_xlabel('fast_adoption_30d')
plt.show()

## Dog vs Cat

In [ ]:
ax = sns.countplot(data=df, x='animal_type', palette='pastel')
ax.set_title('Animal Type Distribution')
plt.show()

## Key Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.countplot(data=df, y='sex_upon_intake', order=df['sex_upon_intake'].value_counts().index[:8], ax=axes[0, 0], palette='Set3')
axes[0, 0].set_title('Sex Upon Intake')

sns.countplot(data=df, y='intake_type', order=df['intake_type'].value_counts().index[:8], ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Intake Type')

sns.countplot(data=df, y='intake_condition', order=df['intake_condition'].value_counts().index[:8], ax=axes[1, 0], palette='Set1')
axes[1, 0].set_title('Intake Condition')

sns.histplot(data=df, x='age_upon_intake_days', bins=30, ax=axes[1, 1], color='#457B9D')
axes[1, 1].set_title('Age Upon Intake (Days)')

plt.tight_layout()
plt.show()

## Relationship With Fast Adoption

In [ ]:
age_plot_df = df.copy()
age_plot_df['age_bin'] = pd.cut(age_plot_df['age_upon_intake_days'], bins=[-1, 90, 365, 3*365, 8*365, 30*365], labels=['0-3m', '3-12m', '1-3y', '3-8y', '8y+'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.barplot(data=age_plot_df, x='age_bin', y='fast_adoption_30d', ax=axes[0, 0], color='#A8DADC')
axes[0, 0].set_title('Fast Adoption Rate by Age Bin')

sns.barplot(data=df, x='animal_type', y='fast_adoption_30d', ax=axes[0, 1], color='#F4A261')
axes[0, 1].set_title('Fast Adoption Rate by Animal Type')

top_breeds = df['breed'].value_counts().index[:10]
breed_df = df[df['breed'].isin(top_breeds)].copy()
sns.barplot(data=breed_df, y='breed', x='fast_adoption_30d', ax=axes[1, 0], color='#2A9D8F')
axes[1, 0].set_title('Fast Adoption Rate by Top Breeds')

sns.barplot(data=df, y='intake_type', x='fast_adoption_30d', order=df['intake_type'].value_counts().index[:8], ax=axes[1, 1], color='#E76F51')
axes[1, 1].set_title('Fast Adoption Rate by Intake Type')

plt.tight_layout()
plt.show()